In [6]:
from bs4 import BeautifulSoup
import pandas as pd
import re
from IPython.display import display


In [ ]:
# load TechPowerUp GPU csv
gpu_df = pd.read_csv("techpowerup_gpus_with_images.csv")

# Advanced Normalization
def normalize_gpu_name(name):
    n = str(name).lower()
    for vendor in ['geforce ', 'radeon ', 'intel ', 'arc ']:
        n = n.replace(vendor, '')
    n = re.sub(r'\s+gb', 'gb', n)
    return n.strip()

# Prepare Search Keys and Sort by Length (Longest first to prevent overlaps like '5060' matching '5060 ti')
gpu_df['Search_Key'] = gpu_df['Name'].apply(normalize_gpu_name)
search_keys_df = gpu_df[['Name', 'Search_Key']].copy()
search_keys_df['Key_Len'] = search_keys_df['Search_Key'].apply(len)
search_keys_df = search_keys_df.sort_values(by='Key_Len', ascending=False)

#  Read the EnterKomputer HTML file 
file_path = "Category_ VGA - Enterkomputer Toko Online Komputer, Simulasi & Rakit PC.html"
with open(file_path, "r", encoding="utf-8") as f:
    html_content = f.read()

soup = BeautifulSoup(html_content, 'html.parser')

# Extract Product Titles and Safe Prices from HTML
scraped_items = []
for prod in soup.find_all('a', class_='ps-product__title'):
    title = prod.text.strip()
    
    parent_div = prod.find_parent('div', class_='ps-product__content')
    if not parent_div: continue
        
    price_p = parent_div.find('p', class_='ps-product__price')
    if not price_p: continue
    
    full_price_text = price_p.get_text(separator=" ", strip=True)
    
    matches = re.findall(r'Rp\s?([\d\.]+)', full_price_text)
    if matches:
        price_str = matches[-1].replace('.', '')
        if price_str.isdigit():
            scraped_items.append({
                'Store_Title': title,
                'Normalized_Title': title.lower().replace(' gb', 'gb'),
                'Price_IDR': int(price_str)
            })

matched_data = []

for item in scraped_items:
    store_title_lower = item['Normalized_Title']
    matched_target_name = None
    
    for _, row in search_keys_df.iterrows():
        key_parts = row['Search_Key'].split()
        if all(part in store_title_lower for part in key_parts):
            matched_target_name = row['Name']
            break
            
    if matched_target_name:
        matched_data.append({
            'GPU_Category': matched_target_name,
            'Vendor/Model_Title': item['Store_Title'],
            'Model_Price_IDR': item['Price_IDR']
        })

df_detailed = pd.DataFrame(matched_data)

if not df_detailed.empty:

    avg_prices = df_detailed.groupby('GPU_Category')['Model_Price_IDR'].mean().round().astype(int).reset_index()
    avg_prices.rename(columns={'Model_Price_IDR': 'Average_Category_Price_IDR'}, inplace=True)
    
    # Merge detailed view with averages
    df_presentation = pd.merge(df_detailed, avg_prices, on='GPU_Category', how='left')
    
    # Sort 
    df_presentation = df_presentation.sort_values(by=['GPU_Category', 'Model_Price_IDR'], ascending=[True, True])
    
    print(f"\n--- DETAILED SCRAPED DATA ({len(df_presentation)} rows) ---")
    display(df_presentation)
    
    # save data to csv
    out_file = "all_scraped_gpu_prices.csv"
    df_presentation.to_csv(out_file, index=False)
    print(f"\nSaved all {len(df_presentation)} scraped rows detailed view to '{out_file}'!")
else:
    print("Warning: No matching prices could be found.")


--- DETAILED SCRAPED DATA (443 rows) ---


,GPU_Category,Vendor/Model_Title,Model_Price_IDR,Average_Category_Price_IDR
277,Arc B580,MAXSUN Intel ARC B580 12GB GDDR6 Milestone 12G...,4950000,5575000
410,Arc B580,SPARKLE Intel ARC B580 12GB GDDR6 Titan OC - B...,4950000,5575000
15,Arc B580,MAXSUN Intel ARC B580 12GB GDDR6 iCraft 12G (W...,5850000,5575000
276,Arc B580,MAXSUN Intel ARC B580 12GB GDDR6 iCraft 12G (W...,5850000,5575000
62,Arc B580,ASRock Intel ARC B580 12GB GDDR6 - Challenger ...,5875000,5575000
...,...,...,...,...
115,Radeon RX 9070 XT,Asus Radeon RX 9070 XT 16GB GDDR6 TUF Gaming O...,14950000,14302368
385,Radeon RX 9070 XT,Sapphire NITRO+ Radeon RX 9070 XT 16GB GDDR6 G...,14950000,14302368
231,Radeon RX 9070 XT,Gigabyte Radeon RX 9070 XT 16GB GDDR6 Aorus El...,15050000,14302368
378,Radeon RX 9070 XT,PowerColor Radeon RX 9070 XT 16GB GDDR6 Hellho...,15395000,14302368



Saved all 443 scraped rows detailed view to 'all_scraped_gpu_prices.csv'!
